
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Tracing for Production Agents
## Overview
Building on basic automatic tracing concepts, this notebook focuses on advanced MLflow Tracing techniques for production agents. You will instrument an agent with manual span decorators and typed spans, attach production-relevant metadata to traces, group multi-turn conversations with session IDs, and search traces programmatically for debugging and analysis.

## Learning Objectives
By the end of this demo, you will be able to:
1. Create manual spans with `@mlflow.trace` and assign semantic span types
2. Attach custom span attributes for production context (user ID, session ID, app version)
3. Group multi-turn conversations using session-level trace attributes
4. Search and filter traces programmatically with `mlflow.search_traces()`

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Install Dependencies and Setup

In [0]:
%run ./Includes/Classroom-Setup-4

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 19.3 MB/s eta 0:00:00
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.2
    Not uninstalling pyyaml at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-0ea8c990-5139-4f0f-9097-9cb42b548206
    Can't uninstall 'PyYAML'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
INFO: pip is looking at multiple versions of unitycatalog-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-openai[databricks] to determine which versio

Starting Demo Environment Setup
Using config: Includes/config/config_4.yaml

[0] Compute check passed: Serverless 5 detected.

[1] Setting up catalog...
Vocareum workspace detected.
  Catalog 'labuser17710504_1793037051' found. Using it.

[2] Setting up schema...
  Catalog : labuser17710504_1793037051
  Schema  : agent_monitoring

[3] Setting up MLflow...
  Setting tracking URI: databricks
Setting up experiment: prod-tracing-training
✅ Created experiment: /Workspace/Users/labuser17710504_1793037051@vocareum.com/prod-tracing-training
  Active experiment: /Workspace/Users/labuser17710504_1793037051@vocareum.com/prod-tracing-training
Refreshing environment

Demo Environment Setup Complete


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


### A1. Quick Recap and Client Setup

Automatic tracing with `mlflow.openai.autolog()` captures LLM calls with zero additional code. In this demo, we go further: manual spans, custom attributes, session grouping, and programmatic trace search.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Recap: Automatic Tracing</strong>
<p style="margin: 8px 0 0 0; color: #333;">
<code>mlflow.openai.autolog()</code> automatically traces OpenAI-compatible LLM calls. Automatic tracing captures the LLM call itself but not custom logic around it. For tool calls, retrieval steps, or business logic, you need <strong>manual tracing</strong>, which is the focus of this demo.
</p>
</div>
</div>
</div>

In [0]:
import mlflow
from openai import OpenAI

# Enable automatic tracing for OpenAI-compatible clients
mlflow.openai.autolog()

DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

BASE_URL = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()+"/serving-endpoints"

client = OpenAI(
    base_url=BASE_URL,
    api_key=DATABRICKS_TOKEN
)

## B. Manual Tracing with Decorators

### B1. Adding Custom Spans

The `span_type` parameter on `@mlflow.trace` is a semantic label that tells MLflow what kind of GenAI step a function represents. This enables structured trace trees, specialized UI rendering, and downstream evaluation scorers that key off span type.

**NOTE:** `span_type` accepts both string literals (e.g., `"TOOL"`) and the `SpanType` enum (e.g., `SpanType.TOOL`).

#### Common values for <code>span_type</code>:

<br></br>

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<style>
.span-row { display: block; white-space: nowrap; font-size: 0; width: 100%; box-sizing: border-box; }

.span-box {
  display: inline-block;
  width: 15.8%;
  margin-right: 1%;
  min-height: 90px;
  background: #F9F7F4;
  border-top: 8px solid transparent;
  border-left: 2px solid transparent;
  border-right: 2px solid transparent;
  border-bottom: 2px solid transparent;
  border-radius: 8px;
  padding: 12px 8px;
  text-align: center;
  box-sizing: border-box;
  cursor: pointer;
  user-select: none;
  vertical-align: top;
  transition: transform 0.12s;
}

.span-box:last-child { margin-right: 0; }

.span-box:hover { transform: translateY(-2px); }

.span-box.active {
  background: #fff;
  border-left-color: var(--pc);
  border-right-color: var(--pc);
  border-bottom-color: var(--pc);
}

.span-title {
  display: block;
  font-size: 13pt;
  font-weight: 700;
  color: #0b2026;
  line-height: 1.3;
  white-space: normal;
  pointer-events: none;
}

.span-detail-wrap {
  overflow: hidden;
  max-height: 0;
  opacity: 0;
  transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease;
  margin-top: 0;
}

.span-detail-wrap.open {
  max-height: 1200px;
  opacity: 1;
  margin-top: 12px;
}

.span-detail-card {
  background: #F9F7F4;
  border-radius: 10px;
  padding: 20px;
  box-sizing: border-box;
  border-top: 7px solid #ccc;
}

.span-can-block {
  border-left: 4px solid;
  border-radius: 0 6px 6px 0;
  padding: 14px 14px;
  margin-bottom: 14px;
  font-size: 14.5pt;
  line-height: 1.6;
}

.span-can-block ul {
  margin: 8px 0 0 0;
  padding-left: 20px;
}

.span-can-block li {
  margin-bottom: 8px;
}

.span-example-block {
  border-left: 4px solid;
  border-radius: 0 6px 6px 0;
  padding: 12px 14px;
  font-size: 14pt;
  line-height: 1.5;
  font-weight: 500;
}
</style>

<div style="background: #F8F9FC; border: 3px solid #1B5162; border-radius: 10px; padding: 24px; box-sizing: border-box;">

  <div class="span-row">
    <span class="span-box active" data-id="0" onclick="selectSpan(0)" style="--pc:#2574B5; border-top-color:#2574B5;">
      <span class="span-title">AGENT</span>
    </span><span class="span-box" data-id="1" onclick="selectSpan(1)" style="--pc:#02A36F; border-top-color:#02A36F;">
      <span class="span-title">TOOL</span>
    </span><span class="span-box" data-id="2" onclick="selectSpan(2)" style="--pc:#FE3722; border-top-color:#FE3722;">
      <span class="span-title">RETRIEVER</span>
    </span><span class="span-box" data-id="3" onclick="selectSpan(3)" style="--pc:#F8A805; border-top-color:#F8A805;">
      <span class="span-title">CHAT_MODEL</span>
    </span><span class="span-box" data-id="4" onclick="selectSpan(4)" style="--pc:#98182A; border-top-color:#98182A;">
      <span class="span-title">PARSER</span>
    </span><span class="span-box" data-id="5" onclick="selectSpan(5)" style="--pc:#1B5162; border-top-color:#1B5162;">
      <span class="span-title">CHAIN</span>
    </span>
  </div>

  <div class="span-detail-wrap" id="span-detail-wrap">
    <div class="span-detail-card" id="span-detail-card"></div>
  </div>

</div>

</div>

<script>
var SPAN_TYPES = [
  {
    title: "AGENT",
    color: "#2574B5",
    bg: "rgba(37,116,181,0.10)",
    exampleBg: "rgba(37,116,181,0.10)",
    content: `
      <ul>
        <li>The <strong>top-level orchestrator</strong> span that represents the overall agent logic</li>
        <li>Typically the root or near-root span in a trace tree, containing child spans for tools, LLM calls, and other steps</li>
        <li>Use this for the main entry point of your agent pipeline</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="AGENT")<br>def travel_agent(user_query: str) -> str:</code>'
  },
  {
    title: "TOOL",
    color: "#02A36F",
    bg: "rgba(2,163,111,0.10)",
    exampleBg: "rgba(2,163,111,0.10)",
    content: `
      <ul>
        <li>Marks a function as a <strong>tool call</strong>: UC functions, SQL queries, API calls, or any utility the agent invokes</li>
        <li>MLflow renders tool spans with dedicated input/output formatting in the trace UI</li>
        <li>Also available as <code>SpanType.TOOL</code></li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="TOOL")<br>def lookup_weather(city: str) -> dict:</code>'
  },
  {
    title: "RETRIEVER",
    color: "#FE3722",
    bg: "rgba(254,55,34,0.10)",
    exampleBg: "rgba(254,55,34,0.10)",
    content: `
      <ul>
        <li>Represents a <strong>retrieval or AI search</strong> step in a RAG pipeline</li>
        <li>Required by evaluation scorers like <strong>RetrievalGroundedness</strong> and <strong>RetrievalRelevance</strong>. These scorers locate RETRIEVER spans to extract retrieved documents</li>
        <li>Use this for any function that fetches context from an AI search index or document retrieval system</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="RETRIEVER")<br>def search_knowledge_base(query: str) -> list:</code>'
  },
  {
    title: "CHAT_MODEL",
    color: "#F8A805",
    bg: "rgba(248,168,5,0.10)",
    exampleBg: "rgba(248,168,5,0.10)",
    content: `
      <ul>
        <li>Represents the underlying <strong>LLM / chat completion call</strong></li>
        <li>Autologging (e.g., <code>mlflow.openai.autolog()</code>) automatically creates CHAT_MODEL spans for supported providers</li>
        <li>Use this when wrapping a custom or direct LLM call that autologging does not capture</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="CHAT_MODEL")<br>def call_llm(messages: list) -> str:</code>'
  },
  {
    title: "PARSER",
    color: "#98182A",
    bg: "rgba(152,24,42,0.10)",
    exampleBg: "rgba(152,24,42,0.10)",
    content: `
      <ul>
        <li>Marks <strong>parsing, formatting, or pre/post-processing</strong> logic</li>
        <li>Use for functions that transform LLM output into structured data, extract fields from responses, or prepare inputs for downstream steps</li>
        <li>Helps distinguish data transformation from business logic in complex traces</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="PARSER")<br>def extract_json_from_response(raw: str) -> dict:</code>'
  },
  {
    title: "CHAIN / FUNCTION",
    color: "#1B5162",
    bg: "rgba(27,81,98,0.10)",
    exampleBg: "rgba(27,81,98,0.10)",
    content: `
      <ul>
        <li>General-purpose span types for <strong>higher-level chains</strong> or <strong>generic helper functions</strong> in a pipeline</li>
        <li><code>CHAIN</code> is useful for grouping a sequence of steps (e.g., retrieve → process → respond) under a single logical unit</li>
        <li><code>FUNCTION</code> is the most generic type. Use it when none of the other span types apply</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="CHAIN")<br>def rag_pipeline(question: str) -> str:</code>'
  }
];

var spanCurrent = null;

function selectSpan(id) {
  var wrap = document.getElementById('span-detail-wrap');
  var card = document.getElementById('span-detail-card');
  var c = SPAN_TYPES[id];

  document.querySelectorAll('.span-box').forEach(function(b) {
    var isActive = parseInt(b.dataset.id, 10) === id;
    b.classList.toggle('active', isActive);
  });

  if (spanCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.span-box').forEach(function(b) {
      b.classList.remove('active');
    });
    spanCurrent = null;
    return;
  }

  spanCurrent = id;

  card.style.borderTopColor = c.color;
  card.innerHTML =
    '<div style="font-size:18pt;font-weight:700;margin-bottom:12px;color:#0b2026;">' + c.title + '</div>'
    + '<div class="span-can-block" style="background:' + c.bg + ';border-color:' + c.color + ';">'
    + c.content
    + '</div>'
    + '<div class="span-example-block" style="background:' + c.exampleBg + ';border-color:' + c.color + ';">'
    + '<strong>Usage:</strong> ' + c.example
    + '</div>';

  wrap.classList.add('open');
}

selectSpan(0);
</script>


#### Select each concept to highlight it in the code

<br></br>

<div style="max-width: 1200px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style>
.c1c-cards { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 18px; }
.c1c-card {
  flex: 1; background: #F9F7F4; border-top: 6px solid transparent;
  border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent;
  border-radius: 8px; padding: 12px 10px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, background 0.15s;
}
.c1c-card:hover { transform: translateY(-2px); }
.c1c-card.active { background: #fff; border-left-color: var(--dc); border-right-color: var(--dc); border-bottom-color: var(--dc); }
.c1c-card-label { display: block; font-size: 12.5pt; font-weight: 700; color: #0b2026; line-height: 1.3; pointer-events: none; }
.c1c-layout { display: flex; gap: 22px; align-items: stretch; }
.c1c-code-wrap { flex: 1; position: relative; }
.c1c-code {
  border: 1px solid #000; border-radius: 10px; padding: 20px 22px; font-family: 'Menlo','Consolas',monospace;
  font-size: 12.5pt; line-height: 1.75; overflow-x: auto; background: #fafafa; color: #383a42;
  transition: background 0.3s, color 0.3s;
}
.c1c-code .tk-kw { color: #a626a4; } .c1c-code .tk-fn { color: #4078f2; }
.c1c-code .tk-str { color: #50a14f; } .c1c-code .tk-num { color: #986801; }
.c1c-code .tk-cmt { color: #a0a1a7; } .c1c-code .tk-dim { color: #696c77; }
.c1c-code .line { display: block; padding: 1px 6px; border-radius: 3px; transition: background 0.25s, opacity 0.25s; }
.c1c-code.has-highlight .line { opacity: 0.35; }
.c1c-code.has-highlight .line.hl { opacity: 1; background: rgba(0,0,0,0.06); }
.c1c-explain { flex: 0 0 260px; display: flex; flex-direction: column; justify-content: center; }
.c1c-explain-card { background: #F9F7F4; border-radius: 10px; border-top: 6px solid #ccc; padding: 20px; font-size: 14pt; line-height: 1.6; opacity: 0; transition: opacity 0.3s; }
.c1c-explain-card.visible { opacity: 1; }
</style>

<div class="c1c-cards">
  <div class="c1c-card" data-id="0" onclick="c1cSelect(0)" style="--dc:#02A36F; border-top-color:#02A36F;">
    <span class="c1c-card-label">TOOL Span</span>
  </div>
  <div class="c1c-card" data-id="1" onclick="c1cSelect(1)" style="--dc:#9B59B6; border-top-color:#9B59B6;">
    <span class="c1c-card-label">CHAIN Span</span>
  </div>
  <div class="c1c-card" data-id="2" onclick="c1cSelect(2)" style="--dc:#F8A805; border-top-color:#F8A805;">
    <span class="c1c-card-label">Agent Orchestration</span>
  </div>
  <div class="c1c-card" data-id="3" onclick="c1cSelect(3)" style="--dc:#4299E0; border-top-color:#4299E0;">
    <span class="c1c-card-label">Invocation</span>
  </div>
</div>

<div class="c1c-layout">
  <div class="c1c-code-wrap" id="c1c-code-wrap">
    <div class="c1c-code" id="c1c-code">
      <span class="line" data-g="0"><span class="tk-fn">@mlflow.trace</span>(<span class="tk-dim">span_type=</span><span class="tk-str">"TOOL"</span>)</span>
      <span class="line" data-g="0"><span class="tk-kw">def</span> <span class="tk-fn">lookup_weather</span>(city: <span class="tk-dim">str</span>) -> <span class="tk-dim">dict</span>:</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt">"""Simulated weather tool for demonstration."""</span></span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;weather_data = {</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"tokyo"</span>: {<span class="tk-str">"temp_f"</span>: <span class="tk-num">72</span>, ...},</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"london"</span>: {<span class="tk-str">"temp_f"</span>: <span class="tk-num">58</span>, ...},</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;}</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">return</span> weather_data.get(city.lower(), ...)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1"><span class="tk-fn">@mlflow.trace</span>(<span class="tk-dim">span_type=</span><span class="tk-str">"CHAIN"</span>)</span>
      <span class="line" data-g="1"><span class="tk-kw">def</span> <span class="tk-fn">travel_agent</span>(user_query: <span class="tk-dim">str</span>) -> <span class="tk-dim">str</span>:</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt">"""A simple agent that looks up weather before responding."""</span></span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt"># Step 1: Determine if weather is needed</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;weather_info = <span class="tk-fn">lookup_weather</span>(<span class="tk-str">"tokyo"</span>)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt"># Step 2: Call the LLM with context</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;response = client.chat.completions.<span class="tk-fn">create</span>(</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;model=<span class="tk-str">"databricks-gpt-5-4"</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;messages=[...],</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;max_tokens=<span class="tk-num">500</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">return</span> response.choices[<span class="tk-num">0</span>].message.content</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">result = <span class="tk-fn">travel_agent</span>(<span class="tk-str">"Should I pack an umbrella for Tokyo?"</span>)</span>
      <span class="line" data-g="3"><span class="tk-fn">print</span>(result)</span>
    </div>
  </div>
  <div class="c1c-explain">
    <div class="c1c-explain-card" id="c1c-explain-card"></div>
  </div>
</div>

</div>

<script>
var C1C_DATA = [
  { color:'#02A36F', title:'@mlflow.trace(span_type="TOOL")',
    text:'Marks <code>lookup_weather</code> as a <strong>TOOL</strong> span. When the agent calls this function, MLflow records it as a distinct tool invocation in the trace tree with its own inputs (city) and outputs (weather dict).' },
  { color:'#9B59B6', title:'@mlflow.trace(span_type="CHAIN")',
    text:'Wraps <code>travel_agent</code> as a <strong>CHAIN</strong> span, grouping a sequence of steps (tool call + LLM call) under a single logical unit. The TOOL span from <code>lookup_weather</code> and the auto-traced CHAT_MODEL span from the LLM call both nest as children.' },
  { color:'#F8A805', title:'Agent Orchestration Logic',
    text:'The body of <code>travel_agent</code> is the orchestration layer: it calls the weather <strong>tool</strong>, injects the result into the prompt, and sends the enriched prompt to the <strong>LLM</strong>. Each step becomes a visible child span.' },
  { color:'#4299E0', title:'Invoking the Agent',
    text:'Calling <code>travel_agent()</code> triggers the full traced execution. The result is a hierarchical trace: <strong>CHAIN</strong> (travel_agent) &rarr; <strong>TOOL</strong> (lookup_weather) + <strong>CHAT_MODEL</strong> (LLM call).' }
];
var c1cCurrent = null, c1cLight = false;
function c1cToggle() {
  c1cLight = !c1cLight;
}
function c1cSelect(id) {
  var code = document.getElementById('c1c-code'), card = document.getElementById('c1c-explain-card');
  document.querySelectorAll('.c1c-card').forEach(function(b){ b.classList.toggle('active', parseInt(b.dataset.id)===id); });
  if (c1cCurrent === id) { code.classList.remove('has-highlight'); card.classList.remove('visible'); document.querySelectorAll('.c1c-card').forEach(function(b){b.classList.remove('active');}); c1cCurrent=null; return; }
  c1cCurrent = id; var c = C1C_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln){ ln.classList.toggle('hl', ln.dataset.g===String(id)); });
  card.style.borderTopColor = c.color;
  card.innerHTML = '<div style="font-size:15pt;font-weight:700;margin-bottom:10px;color:'+c.color+';">'+c.title+'</div><div>'+c.text+'</div>';
  card.classList.add('visible');
}
</script>

In [0]:
import mlflow
import json

@mlflow.trace(span_type="TOOL")
def lookup_weather(city: str) -> dict:
    """Simulated weather tool for demonstration."""
    weather_data = {
        "tokyo": {"temp_f": 72, "condition": "partly cloudy", "humidity": 65},
        "london": {"temp_f": 58, "condition": "rainy", "humidity": 80},
        "new york": {"temp_f": 68, "condition": "sunny", "humidity": 45},
    }
    return weather_data.get(city.lower(), {"temp_f": 0, "condition": "unknown", "humidity": 0})


@mlflow.trace(span_type="CHAIN")
def travel_agent(user_query: str) -> str:
    """A simple agent that looks up weather before responding."""
    # Step 1: Determine if weather is needed
    weather_info = lookup_weather("tokyo")

    # Step 2: Call the LLM with context
    response = client.chat.completions.create(
        model="databricks-gpt-5-4",
        messages=[
            {"role": "system", "content": "You are a travel assistant. Use the provided weather data in your response."},
            {"role": "user", "content": f"{user_query}\n\nCurrent weather: {json.dumps(weather_info)}"}
        ],
        max_tokens=500
    )

    return response.choices[0].message.content


result = travel_agent("Should I pack an umbrella for Tokyo this weekend?")
print(result)

Maybe—but I wouldn’t make it a must-pack based on the weather you shared.

For Tokyo, the current weather is **72°F, partly cloudy, with 65% humidity**, which doesn’t strongly suggest rain right now. An umbrella could still be useful if the forecast changes, but based only on this data, **it’s more of a “nice to have” than essential**.

If you want, I can also help you decide what else to pack for Tokyo this weekend.


Trace(trace_id=tr-21df97a4fd719dc152fcae3962bbf7f3)

### B2. Reading the Span Tree

The trace now shows a hierarchy. Each span records its start time, end time, inputs, outputs, and status.

<div style="max-width:680px;margin:16px 0;font-family:'Menlo','Consolas',monospace;font-size:13pt;line-height:2;background:#f7f8fc;border:1px solid #dde0e8;border-radius:10px;padding:18px 22px;">
<div style="color:#3b82f6;font-weight:700;">CHAIN <span style="color:#999;font-weight:400;">travel_agent</span></div>
<div style="margin-left:24px;border-left:2px solid #dde0e8;padding-left:14px;">
<div style="color:#16a34a;font-weight:700;">TOOL <span style="color:#999;font-weight:400;">lookup_weather</span> <span style="color:#bbb;font-size:11pt;">inputs: "tokyo" &rarr; outputs: {temp_f: 72, ...}</span></div>
<div style="color:#a855f7;font-weight:700;">CHAT_MODEL <span style="color:#999;font-weight:400;">auto-captured</span> <span style="color:#bbb;font-size:11pt;">prompt + response</span></div>
</div>
</div>

This is the execution graph you will use for debugging in Module 3.

## C. Span Attributes for Production Context

### C1. Adding Metadata to Spans

In production, you need to correlate traces with business context: which user made the request, what session it belongs to, what application version is running. Span attributes let you attach arbitrary key-value metadata to any span.

For example, after running the cell below, click on **Attributes** to view the different attributes that were logged with `span.set_attributes()` in the definition of `travel_agent_with_context()`.


#### Select each concept to highlight it in the code

<br></br>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style>
.d1-cards {
  display: flex;
  flex-wrap: wrap;
  gap: 8px;
  margin-bottom: 18px;
}

.d1-card {
  flex: 1;
  background: #F9F7F4;
  border-top: 6px solid transparent;
  border-left: 2px solid transparent;
  border-right: 2px solid transparent;
  border-bottom: 2px solid transparent;
  border-radius: 8px;
  padding: 12px 10px;
  text-align: center;
  cursor: pointer;
  user-select: none;
  transition: transform 0.12s, background 0.15s;
}

.d1-card:hover { transform: translateY(-2px); }

.d1-card.active {
  background: #fff;
  border-left-color: var(--dc);
  border-right-color: var(--dc);
  border-bottom-color: var(--dc);
}

.d1-card-label {
  display: block;
  font-size: 12.5pt;
  font-weight: 700;
  color: #0b2026;
  line-height: 1.3;
  pointer-events: none;
}

.d1-layout {
  display: flex;
  gap: 22px;
  align-items: stretch;
}

.d1-code-wrap {
  flex: 1;
  position: relative;
}

.d1-code {
  border: 1px solid #000; border-radius: 10px;
  padding: 20px 22px;
  font-family: 'Menlo', 'Consolas', monospace;
  font-size: 12.5pt;
  line-height: 1.75;
  overflow-x: auto;
  background: #fafafa;
  color: #383a42;
}

.d1-code .tk-kw { color: #a626a4; }
.d1-code .tk-fn { color: #4078f2; }
.d1-code .tk-str { color: #50a14f; }
.d1-code .tk-num { color: #986801; }
.d1-code .tk-cmt { color: #a0a1a7; }
.d1-code .tk-dim { color: #696c77; }

.d1-code .line {
  display: block;
  padding: 1px 6px;
  border-radius: 3px;
  transition: background 0.25s, opacity 0.25s;
}

.d1-code.has-highlight .line { opacity: 0.35; }
.d1-code.has-highlight .line.hl { opacity: 1; background: rgba(0,0,0,0.06); }

.d1-explain {
  flex: 0 0 300px;
  display: flex;
  flex-direction: column;
  justify-content: center;
}

.d1-explain-card {
  background: #F9F7F4;
  border-radius: 10px;
  border-top: 6px solid #ccc;
  padding: 20px;
  font-size: 14pt;
  line-height: 1.6;
  opacity: 0;
  transition: opacity 0.3s;
}

.d1-explain-card.visible { opacity: 1; }
</style>

<!-- Card selectors -->
<div class="d1-cards">
  <div class="d1-card" data-id="0" onclick="d1Select(0)" style="--dc:#9B59B6; border-top-color:#9B59B6;">
    <span class="d1-card-label">@mlflow.trace</span>
  </div>
  <div class="d1-card" data-id="1" onclick="d1Select(1)" style="--dc:#4299E0; border-top-color:#4299E0;">
    <span class="d1-card-label">Get Active Span</span>
  </div>
  <div class="d1-card" data-id="2" onclick="d1Select(2)" style="--dc:#02A36F; border-top-color:#02A36F;">
    <span class="d1-card-label">Set Attributes</span>
  </div>
  <div class="d1-card" data-id="3" onclick="d1Select(3)" style="--dc:#F8A805; border-top-color:#F8A805;">
    <span class="d1-card-label">Tool + LLM</span>
  </div>
</div>

<!-- Code + explanation side by side -->
<div class="d1-layout">

  <div class="d1-code-wrap" id="d1-code-wrap">
    <div class="d1-code" id="d1-code">
      <span class="line" data-g="0"><span class="tk-fn">@mlflow.trace</span>(<span class="tk-dim">span_type=</span><span class="tk-str">"CHAIN"</span>)</span>
      <span class="line" data-g="0"><span class="tk-kw">def</span> <span class="tk-fn">travel_agent_with_context</span>(user_query, user_id, session_id):</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt">"""Agent with production context attached to the trace."""</span></span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt"># Get the current span and add production metadata</span></span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;span = mlflow.<span class="tk-fn">get_current_active_span</span>()</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;span.<span class="tk-fn">set_attributes</span>({</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"user_id"</span>: user_id,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"session_id"</span>: session_id,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"app_version"</span>: <span class="tk-str">"1.2.0"</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"request_source"</span>: <span class="tk-str">"web_ui"</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;})</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;weather_info = <span class="tk-fn">lookup_weather</span>(<span class="tk-str">"london"</span>)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;response = client.chat.completions.<span class="tk-fn">create</span>(</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;model=<span class="tk-str">"databricks-gpt-5-4"</span>,</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;messages=[...],</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;max_tokens=<span class="tk-num">500</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;)</span>
      <span class="line">&nbsp;</span>
      <span class="line">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">return</span> response.choices[<span class="tk-num">0</span>].message.content</span>
    </div>
  </div>

  <div class="d1-explain">
    <div class="d1-explain-card" id="d1-explain-card"></div>
  </div>

</div>

</div>

<script>
var D1_DATA = [
  {
    color: '#9B59B6',
    title: '@mlflow.trace(span_type="CHAIN")',
    text: 'The decorator wraps the entire function as a <strong>CHAIN</strong> span. Everything that executes inside this function becomes a child in the trace tree. The <code>span_type</code> label tells MLflow this is a higher-level orchestration step, not a tool or LLM call.'
  },
  {
    color: '#4299E0',
    title: 'get_current_active_span()',
    text: 'Retrieves a reference to the span that <code>@mlflow.trace</code> just created. This gives you a handle to enrich the span from <em>inside</em> the function body, rather than only relying on what the decorator captures automatically (inputs and outputs).'
  },
  {
    color: '#02A36F',
    title: 'set_attributes({...})',
    text: 'Attaches arbitrary key-value pairs to the span. These attributes (<code>user_id</code>, <code>session_id</code>, <code>app_version</code>, <code>request_source</code>) become <strong>searchable and filterable</strong> in the MLflow UI and via <code>mlflow.search_traces()</code>.'
  },
  {
    color: '#F8A805',
    title: 'Tool Call + LLM Request',
    text: 'The <code>lookup_weather</code> tool call and <code>chat.completions.create</code> LLM call execute as normal Python. Because autologging is enabled, MLflow automatically creates a <strong>TOOL</strong> child span and a <strong>CHAT_MODEL</strong> child span nested under this CHAIN.'
  }
];

var d1Current = null;

function d1Select(id) {
  var codeEl = document.getElementById('d1-code');
  var explainCard = document.getElementById('d1-explain-card');

  // Toggle cards
  document.querySelectorAll('.d1-card').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id);
  });

  // Deselect if same card clicked
  if (d1Current === id) {
    codeEl.classList.remove('has-highlight');
    explainCard.classList.remove('visible');
    document.querySelectorAll('.d1-card').forEach(function(b) { b.classList.remove('active'); });
    d1Current = null;
    return;
  }

  d1Current = id;
  var c = D1_DATA[id];

  // Highlight matching code lines
  codeEl.classList.add('has-highlight');
  codeEl.querySelectorAll('.line').forEach(function(ln) {
    ln.classList.toggle('hl', ln.dataset.g === String(id));
  });

  // Show explanation
  explainCard.style.borderTopColor = c.color;
  explainCard.innerHTML =
    '<div style="font-size:15pt;font-weight:700;margin-bottom:10px;color:' + c.color + ';">' + c.title + '</div>'
    + '<div>' + c.text + '</div>';
  explainCard.classList.add('visible');
}
</script>

In [0]:
@mlflow.trace(span_type="CHAIN")
def travel_agent_with_context(user_query: str, user_id: str, session_id: str) -> str:
    """Agent with production context attached to the trace."""
    # Get the current span and add production metadata
    span = mlflow.get_current_active_span()
    span.set_attributes({
        "user_id": user_id,
        "session_id": session_id,
        "app_version": "1.2.0",
        "request_source": "web_ui"
    })

    weather_info = lookup_weather("london")

    response = client.chat.completions.create(
        model="databricks-gpt-5-4",
        messages=[
            {"role": "system", "content": "You are a travel assistant."},
            {"role": "user", "content": f"{user_query}\n\nWeather: {json.dumps(weather_info)}"}
        ],
        max_tokens=500
    )

    return response.choices[0].message.content


result = travel_agent_with_context(
    user_query="What should I wear in London tomorrow?",
    user_id="user_42",
    session_id="sess_abc123"
)
print(result)

Wear light layers and plan for rain.

Suggested outfit:
- Waterproof jacket or raincoat
- Long-sleeve shirt or light sweater
- Jeans or other long pants
- Waterproof shoes or sneakers
- Umbrella

At 58°F, it’ll feel cool, and with rain and high humidity it may feel a bit chilly and damp. If you’ll be out in the evening, add an extra layer.


Trace(trace_id=tr-617e414d8eae051425a821ebbc913dc3)

### C2. Why Attributes Matter

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Span attributes are searchable and filterable. You can use attributes to slice traces by user, session, or app version when debugging production issues. Attach them early as you cannot add them retroactively.</p>
</div>
</div>
</div>

## D. Multi-Turn Conversation Tracing

### D1. Session-Level Grouping

Production agents often handle multi-turn conversations. Each turn produces its own trace, but you need to group them into sessions for analysis. Use a consistent `session_id` attribute across turns to enable session-level queries.


#### Select each concept to highlight it in the code

<br></br>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style>
.e1-cards { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 18px; }
.e1-card {
  flex: 1; background: #F9F7F4; border-top: 6px solid transparent;
  border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent;
  border-radius: 8px; padding: 12px 10px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, background 0.15s;
}
.e1-card:hover { transform: translateY(-2px); }
.e1-card.active { background: #fff; border-left-color: var(--dc); border-right-color: var(--dc); border-bottom-color: var(--dc); }
.e1-card-label { display: block; font-size: 12.5pt; font-weight: 700; color: #0b2026; line-height: 1.3; pointer-events: none; }
.e1-layout { display: flex; gap: 22px; align-items: stretch; }
.e1-code-wrap { flex: 1; position: relative; }
.e1-code {
  border: 1px solid #000; border-radius: 10px; padding: 20px 22px; font-family: 'Menlo','Consolas',monospace;
  font-size: 12.5pt; line-height: 1.75; overflow-x: auto; background: #fafafa; color: #383a42;
  transition: background 0.3s, color 0.3s;
}
.e1-code .tk-kw { color: #a626a4; } .e1-code .tk-fn { color: #4078f2; }
.e1-code .tk-str { color: #50a14f; } .e1-code .tk-num { color: #986801; }
.e1-code .tk-cmt { color: #a0a1a7; } .e1-code .tk-dim { color: #696c77; }
.e1-code .line { display: block; padding: 1px 6px; border-radius: 3px; transition: background 0.25s, opacity 0.25s; }
.e1-code.has-highlight .line { opacity: 0.35; }
.e1-code.has-highlight .line.hl { opacity: 1; background: rgba(0,0,0,0.06); }
.e1-explain { flex: 0 0 300px; display: flex; flex-direction: column; justify-content: center; }
.e1-explain-card { background: #F9F7F4; border-radius: 10px; border-top: 6px solid #ccc; padding: 20px; font-size: 14pt; line-height: 1.6; opacity: 0; transition: opacity 0.3s; }
.e1-explain-card.visible { opacity: 1; }
</style>

<div class="e1-cards">
  <div class="e1-card" data-id="0" onclick="e1Select(0)" style="--dc:#9B59B6; border-top-color:#9B59B6;">
    <span class="e1-card-label">Session Setup</span>
  </div>
  <div class="e1-card" data-id="1" onclick="e1Select(1)" style="--dc:#4299E0; border-top-color:#4299E0;">
    <span class="e1-card-label">Turn Decorator</span>
  </div>
  <div class="e1-card" data-id="2" onclick="e1Select(2)" style="--dc:#02A36F; border-top-color:#02A36F;">
    <span class="e1-card-label">Session Attributes</span>
  </div>
  <div class="e1-card" data-id="3" onclick="e1Select(3)" style="--dc:#F8A805; border-top-color:#F8A805;">
    <span class="e1-card-label">Conversation Memory</span>
  </div>
</div>

<div class="e1-layout">
  <div class="e1-code-wrap" id="e1-code-wrap">
    <div class="e1-code" id="e1-code">
      <span class="line" data-g="0"><span class="tk-kw">import</span> uuid</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="0">session_id = <span class="tk-fn">str</span>(<span class="tk-fn">uuid.uuid4</span>())</span>
      <span class="line" data-g="0">conversation_history = [</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;{<span class="tk-str">"role"</span>: <span class="tk-str">"system"</span>, <span class="tk-str">"content"</span>: <span class="tk-str">"You are a travel assistant. Be concise."</span>}</span>
      <span class="line" data-g="0">]</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1"><span class="tk-fn">@mlflow.trace</span>(<span class="tk-dim">span_type=</span><span class="tk-str">"CHAIN"</span>, <span class="tk-dim">name=</span><span class="tk-str">"conversation_turn"</span>)</span>
      <span class="line" data-g="1"><span class="tk-kw">def</span> <span class="tk-fn">chat_turn</span>(user_message: <span class="tk-dim">str</span>, turn_number: <span class="tk-dim">int</span>) -> <span class="tk-dim">str</span>:</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;span = mlflow.<span class="tk-fn">get_current_active_span</span>()</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;span.<span class="tk-fn">set_attributes</span>({</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"session_id"</span>: session_id,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"turn_number"</span>: turn_number</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;})</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;conversation_history.<span class="tk-fn">append</span>({<span class="tk-str">"role"</span>: <span class="tk-str">"user"</span>, <span class="tk-str">"content"</span>: user_message})</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;response = client.chat.completions.<span class="tk-fn">create</span>(</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;model=<span class="tk-str">"databricks-gpt-5-4"</span>,</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;messages=conversation_history,</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;max_tokens=<span class="tk-num">300</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;assistant_msg = response.choices[<span class="tk-num">0</span>].message.content</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;conversation_history.<span class="tk-fn">append</span>({<span class="tk-str">"role"</span>: <span class="tk-str">"assistant"</span>, <span class="tk-str">"content"</span>: assistant_msg})</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">return</span> assistant_msg</span>
    </div>
  </div>
  <div class="e1-explain">
    <div class="e1-explain-card" id="e1-explain-card"></div>
  </div>
</div>

</div>

<script>
var E1_DATA = [
  { color:'#9B59B6', title:'Session Setup',
    text:'A single <code>session_id</code> (UUID) is generated once and shared across all turns. The <code>conversation_history</code> list accumulates messages, giving the LLM context from prior turns.' },
  { color:'#4299E0', title:'Decorator with name= Parameter',
    text:'<code>@mlflow.trace(name="conversation_turn")</code> gives the span a <strong>custom name</strong> instead of defaulting to the function name. This makes traces easier to identify when browsing the UI.' },
  { color:'#02A36F', title:'Session & Turn Attributes',
    text:'Each turn attaches <code>session_id</code> and <code>turn_number</code> as span attributes. This lets you <strong>group all turns in a session</strong> with <code>mlflow.search_traces()</code> and filter by turn order.' },
  { color:'#F8A805', title:'Conversation Memory & LLM Call',
    text:'The user message is appended to history, the full history is sent to the LLM, and the assistant response is appended back. This accumulating pattern is how multi-turn context is maintained. The LLM call is auto-traced as a <strong>CHAT_MODEL</strong> child span.' }
];
var e1Current = null, e1Light = false;
function e1Toggle() {
  e1Light = !e1Light;
}
function e1Select(id) {
  var code = document.getElementById('e1-code'), card = document.getElementById('e1-explain-card');
  document.querySelectorAll('.e1-card').forEach(function(b){ b.classList.toggle('active', parseInt(b.dataset.id)===id); });
  if (e1Current === id) { code.classList.remove('has-highlight'); card.classList.remove('visible'); document.querySelectorAll('.e1-card').forEach(function(b){b.classList.remove('active');}); e1Current=null; return; }
  e1Current = id; var c = E1_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln){ ln.classList.toggle('hl', ln.dataset.g===String(id)); });
  card.style.borderTopColor = c.color;
  card.innerHTML = '<div style="font-size:15pt;font-weight:700;margin-bottom:10px;color:'+c.color+';">'+c.title+'</div><div>'+c.text+'</div>';
  card.classList.add('visible');
}
</script>

In [0]:
import uuid

session_id = str(uuid.uuid4())
conversation_history = [
    {"role": "system", "content": "You are a travel assistant. Be concise."}
]


@mlflow.trace(span_type="CHAIN", name="conversation_turn")
def chat_turn(user_message: str, turn_number: int) -> str:
    """Process a single turn in a multi-turn conversation."""
    span = mlflow.get_current_active_span()
    span.set_attributes({
        "session_id": session_id,
        "turn_number": turn_number
    })

    conversation_history.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model="databricks-gpt-5-4",
        messages=conversation_history,
        max_tokens=300
    )

    assistant_msg = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": assistant_msg})
    return assistant_msg


# Simulate a multi-turn conversation
print("Turn 1:", chat_turn("I'm planning a trip to Japan in April.", turn_number=1))

Turn 1: Nice choice—April is one of the best times to visit Japan because of spring weather and cherry blossoms.

A few quick questions so I can help well:
1. How long is your trip?
2. What’s your budget range?
3. First time in Japan?
4. Do you want more cities, food, nature, anime, or traditional culture?
5. Are you thinking Tokyo/Kyoto/Osaka, or open to other regions too?

In the meantime, here are key April notes:
- Weather: Mild, generally around 10–20°C depending on region.
- Cherry blossoms: Usually peak in many major cities in late March to early April, but timing changes each year.
- Crowds: Very busy in popular spots, especially during blossom season.
- What to book early: Flights, hotels, and popular attractions.

Classic first-trip route:
- Tokyo: Modern city, shopping, food, neighborhoods
- Kyoto: Temples, gardens, traditional atmosphere
- Osaka: Street food, nightlife, day trips
- Optional: Nara, Hakone, Hiroshima, Miyajima

If you want, I can build you:
- a 7-day itinerar

Trace(trace_id=tr-5857acbc702bbdf7f828e251f407b5ad)

In [0]:
print("Turn 2:", chat_turn("What about the cherry blossoms, will I catch them?", turn_number=2))

Turn 2: Possibly—April can be great, but it depends on **exact dates** and **where in Japan**.

Typical timing:
- **Tokyo / Kyoto / Osaka:** often **late March to early April**
- **Higher elevations** (Hakone, Nikko, Japanese Alps): usually **a bit later**
- **Tohoku** (north Honshu): often **mid to late April**
- **Hokkaido:** usually **late April to early May**

So:
- **Early April:** good chance in Tokyo, Kyoto, Osaka
- **Mid-April:** blossoms may be past peak in those cities, but you can still catch them farther north or in cooler areas
- **Late April:** better to target Tohoku or Hokkaido

A few important notes:
- Peak bloom only lasts a short time, often about **a week**, and weather can shift it
- Even if peak has passed, you may still see **some blossoms** or beautiful **falling petals**

If you tell me your **travel dates**, I can suggest the best places in Japan to maximize your cherry blossom chances.


Trace(trace_id=tr-88906d97d1894917d7c38bb048e00dd4)

In [0]:
print("Turn 3:", chat_turn("Do you recommend anything else to do while in Kyoto?", turn_number=3))

Turn 3: Yes—Kyoto has a lot beyond the most famous temples. Good options:

### Classic highlights
- **Fushimi Inari Taisha** — go early or at night for fewer crowds
- **Kiyomizu-dera** — combine with **Sannenzaka** and **Ninenzaka**
- **Arashiyama** — bamboo grove, river area, monkey park
- **Kinkaku-ji** — iconic, quick visit

### Areas to wander
- **Gion** — traditional streets, especially in the evening
- **Higashiyama** — best for atmosphere and old Kyoto feel
- **Pontocho Alley** — great for dinner
- **Nishiki Market** — snacks and local food

### Less crowded / very worthwhile
- **Philosopher’s Path** — especially nice in spring
- **Nanzen-ji** — spacious temple grounds
- **Eikan-do** — beautiful gardens
- **Shimogamo Shrine** or **Kamigamo Shrine** — calmer and historic

### Cultural experiences
- **Tea ceremony**
- **Kaiseki dinner**
- **Kimono rental** for walking around old districts
- **Zen meditation** or temple stay experience

### Easy day trips from Kyoto
- **Nara** — de

Trace(trace_id=tr-cce534b68b14a355f18f3b8d9727631d)

### D2. Viewing Session Traces

Each turn above produced a separate trace, but they all share the same `session_id` attribute. This lets you reconstruct the full conversation during debugging or analysis.

## E. Searching and Filtering Traces

### E1. Programmatic Trace Search

In production, you will not browse traces one at a time in the UI. `mlflow.search_traces()` lets you query traces programmatically. Filter by time range, status, attributes, or any combination.


#### Select each concept to highlight it in the code

<br></br>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style>
.f1-cards { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 18px; }
.f1-card {
  flex: 1; background: #F9F7F4; border-top: 6px solid transparent;
  border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent;
  border-radius: 8px; padding: 12px 10px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, background 0.15s;
}
.f1-card:hover { transform: translateY(-2px); }
.f1-card.active { background: #fff; border-left-color: var(--dc); border-right-color: var(--dc); border-bottom-color: var(--dc); }
.f1-card-label { display: block; font-size: 12.5pt; font-weight: 700; color: #0b2026; line-height: 1.3; pointer-events: none; }
.f1-layout { display: flex; gap: 22px; align-items: stretch; }
.f1-code-wrap { flex: 1; position: relative; }
.f1-code {
  border: 1px solid #000; border-radius: 10px; padding: 20px 22px; font-family: 'Menlo','Consolas',monospace;
  font-size: 12.5pt; line-height: 1.75; overflow-x: auto; background: #fafafa; color: #383a42;
  transition: background 0.3s, color 0.3s;
}
.f1-code .tk-kw { color: #a626a4; } .f1-code .tk-fn { color: #4078f2; }
.f1-code .tk-str { color: #50a14f; } .f1-code .tk-num { color: #986801; }
.f1-code .tk-cmt { color: #a0a1a7; } .f1-code .tk-dim { color: #696c77; }
.f1-code .line { display: block; padding: 1px 6px; border-radius: 3px; transition: background 0.25s, opacity 0.25s; }
.f1-code.has-highlight .line { opacity: 0.35; }
.f1-code.has-highlight .line.hl { opacity: 1; background: rgba(0,0,0,0.06); }
.f1-explain { flex: 0 0 300px; display: flex; flex-direction: column; justify-content: center; }
.f1-explain-card { background: #F9F7F4; border-radius: 10px; border-top: 6px solid #ccc; padding: 20px; font-size: 14pt; line-height: 1.6; opacity: 0; transition: opacity 0.3s; }
.f1-explain-card.visible { opacity: 1; }
</style>

<div class="f1-cards">
  <div class="f1-card" data-id="0" onclick="f1Select(0)" style="--dc:#9B59B6; border-top-color:#9B59B6;">
    <span class="f1-card-label">Get Experiment</span>
  </div>
  <div class="f1-card" data-id="1" onclick="f1Select(1)" style="--dc:#02A36F; border-top-color:#02A36F;">
    <span class="f1-card-label">Search Traces</span>
  </div>
  <div class="f1-card" data-id="2" onclick="f1Select(2)" style="--dc:#4299E0; border-top-color:#4299E0;">
    <span class="f1-card-label">Display Results</span>
  </div>
</div>

<div class="f1-layout">
  <div class="f1-code-wrap" id="f1-code-wrap">
    <div class="f1-code" id="f1-code">
      <span class="line" data-g="0"><span class="tk-cmt"># Get the experiment associated with this notebook</span></span>
      <span class="line" data-g="0">experiment = mlflow.<span class="tk-fn">get_experiment_by_name</span>(</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;mlflow_experiment_path</span>
      <span class="line" data-g="0">)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="0"><span class="tk-cmt"># If no experiment found, use the active one</span></span>
      <span class="line" data-g="0">experiment_id = experiment.experiment_id \</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">if</span> experiment <span class="tk-kw">else</span> mlflow.<span class="tk-fn">get_experiment</span>(<span class="tk-num">0</span>).experiment_id</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1"><span class="tk-cmt"># Search for recent traces</span></span>
      <span class="line" data-g="1">traces_df = mlflow.<span class="tk-fn">search_traces</span>(</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;locations=[experiment_id],</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;max_results=<span class="tk-num">10</span></span>
      <span class="line" data-g="1">)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2"><span class="tk-fn">display</span>(traces_df[[</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"client_request_id"</span>, <span class="tk-str">"trace_id"</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"request_time"</span>, <span class="tk-str">"state"</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"execution_duration"</span></span>
      <span class="line" data-g="2">]])</span>
    </div>
  </div>
  <div class="f1-explain">
    <div class="f1-explain-card" id="f1-explain-card"></div>
  </div>
</div>

</div>

<script>
var F1_DATA = [
  { color:'#9B59B6', title:'Resolve the Experiment',
    text:'Looks up the MLflow experiment by its workspace path. Every notebook has an associated experiment where traces and runs are logged. The fallback ensures the code works even if the path variable is not set.' },
  { color:'#02A36F', title:'mlflow.search_traces()',
    text:'The core query API. Takes a <code>locations</code> list (experiment IDs) and optional <code>filter_string</code>, <code>max_results</code>, and <code>order_by</code> parameters. Filter on <code>trace.*</code> (status, timing), <code>metadata.*</code>, <code>tag.*</code>, or <code>span.attributes.*</code>. Returns a <strong>Pandas DataFrame</strong> with one row per trace.' },
  { color:'#4299E0', title:'Display Selected Columns',
    text:'Selects the most useful columns for a quick overview: <code>client_request_id</code>, <code>trace_id</code>, <code>request_time</code>, <code>state</code> (OK or ERROR), and <code>execution_duration</code>. The Databricks <code>display()</code> renders them as an interactive table.' }
];
var f1Current = null, f1Light = false;
function f1Toggle() {
  f1Light = !f1Light;
}
function f1Select(id) {
  var code = document.getElementById('f1-code'), card = document.getElementById('f1-explain-card');
  document.querySelectorAll('.f1-card').forEach(function(b){ b.classList.toggle('active', parseInt(b.dataset.id)===id); });
  if (f1Current === id) { code.classList.remove('has-highlight'); card.classList.remove('visible'); document.querySelectorAll('.f1-card').forEach(function(b){b.classList.remove('active');}); f1Current=null; return; }
  f1Current = id; var c = F1_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln){ ln.classList.toggle('hl', ln.dataset.g===String(id)); });
  card.style.borderTopColor = c.color;
  card.innerHTML = '<div style="font-size:15pt;font-weight:700;margin-bottom:10px;color:'+c.color+';">'+c.title+'</div><div>'+c.text+'</div>';
  card.classList.add('visible');
}
</script>

In [0]:
# Get the experiment associated with this notebook
experiment = mlflow.get_experiment_by_name(
    mlflow_experiment_path
)

# If no experiment found, use the active one
experiment_id = experiment.experiment_id if experiment else mlflow.get_experiment(0).experiment_id

# Search for recent traces. Adjust filter as needed
traces_df = mlflow.search_traces(
    locations=[experiment_id],
    max_results=10
)

display(traces_df[["client_request_id", "trace_id", "request_time", "state", "execution_duration"]])

client_request_id,trace_id,request_time,state,execution_duration
tr-cce534b68b14a355f18f3b8d9727631d,tr-cce534b68b14a355f18f3b8d9727631d,1793041657563,OK,7819
tr-88906d97d1894917d7c38bb048e00dd4,tr-88906d97d1894917d7c38bb048e00dd4,1793041636942,OK,5000
tr-5857acbc702bbdf7f828e251f407b5ad,tr-5857acbc702bbdf7f828e251f407b5ad,1793041610911,OK,6197
tr-617e414d8eae051425a821ebbc913dc3,tr-617e414d8eae051425a821ebbc913dc3,1793040517045,OK,2688
tr-21df97a4fd719dc152fcae3962bbf7f3,tr-21df97a4fd719dc152fcae3962bbf7f3,1793039568974,OK,4759
tr-b80180f6988b959a1c6b48c89b39a829,tr-b80180f6988b959a1c6b48c89b39a829,1793039537108,ERROR,3


### E2. Filtering by Status and Timing

Common production queries include finding error traces and slow traces.


#### Select each concept to highlight it in the code

<br></br>

<div style="max-width: 1060px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style>
.f2-cards { display: flex; flex-wrap: wrap; gap: 8px; margin-bottom: 18px; }
.f2-card {
  flex: 1; background: #F9F7F4; border-top: 6px solid transparent;
  border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent;
  border-radius: 8px; padding: 12px 10px; text-align: center; cursor: pointer; user-select: none;
  transition: transform 0.12s, background 0.15s;
}
.f2-card:hover { transform: translateY(-2px); }
.f2-card.active { background: #fff; border-left-color: var(--dc); border-right-color: var(--dc); border-bottom-color: var(--dc); }
.f2-card-label { display: block; font-size: 12.5pt; font-weight: 700; color: #0b2026; line-height: 1.3; pointer-events: none; }
.f2-layout { display: flex; gap: 22px; align-items: stretch; }
.f2-code-wrap { flex: 1; position: relative; }
.f2-code {
  border: 1px solid #000; border-radius: 10px; padding: 20px 22px; font-family: 'Menlo','Consolas',monospace;
  font-size: 12.5pt; line-height: 1.75; overflow-x: auto; background: #fafafa; color: #383a42;
  transition: background 0.3s, color 0.3s;
}
.f2-code .tk-kw { color: #a626a4; } .f2-code .tk-fn { color: #4078f2; }
.f2-code .tk-str { color: #50a14f; } .f2-code .tk-num { color: #986801; }
.f2-code .tk-cmt { color: #a0a1a7; } .f2-code .tk-dim { color: #696c77; }
.f2-code .line { display: block; padding: 1px 6px; border-radius: 3px; transition: background 0.25s, opacity 0.25s; }
.f2-code.has-highlight .line { opacity: 0.35; }
.f2-code.has-highlight .line.hl { opacity: 1; background: rgba(0,0,0,0.06); }
.f2-explain { flex: 0 0 300px; display: flex; flex-direction: column; justify-content: center; }
.f2-explain-card { background: #F9F7F4; border-radius: 10px; border-top: 6px solid #ccc; padding: 20px; font-size: 14pt; line-height: 1.6; opacity: 0; transition: opacity 0.3s; }
.f2-explain-card.visible { opacity: 1; }
</style>

<div class="f2-cards">
  <div class="f2-card" data-id="0" onclick="f2Select(0)" style="--dc:#FE3722; border-top-color:#FE3722;">
    <span class="f2-card-label">Error Traces</span>
  </div>
  <div class="f2-card" data-id="1" onclick="f2Select(1)" style="--dc:#F8A805; border-top-color:#F8A805;">
    <span class="f2-card-label">Slow Traces</span>
  </div>
</div>

<div class="f2-layout">
  <div class="f2-code-wrap" id="f2-code-wrap">
    <div class="f2-code" id="f2-code">
      <span class="line" data-g="0"><span class="tk-cmt"># Find error traces</span></span>
      <span class="line" data-g="0">error_traces = mlflow.<span class="tk-fn">search_traces</span>(</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;locations=[experiment_id],</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;filter_string=<span class="tk-str">"trace.status = 'ERROR'"</span>,</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;max_results=<span class="tk-num">100</span></span>
      <span class="line" data-g="0">)</span>
      <span class="line" data-g="0"><span class="tk-fn">print</span>(<span class="tk-str">f"Error traces found: {</span><span class="tk-fn">len</span>(error_traces)<span class="tk-str">}"</span>)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1"><span class="tk-cmt"># Find slow traces (over 5 seconds)</span></span>
      <span class="line" data-g="1">slow_traces = mlflow.<span class="tk-fn">search_traces</span>(</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;locations=[experiment_id],</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;filter_string=<span class="tk-str">"trace.execution_time_ms > 5000"</span>,</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;max_results=<span class="tk-num">100</span></span>
      <span class="line" data-g="1">)</span>
      <span class="line" data-g="1"><span class="tk-fn">print</span>(<span class="tk-str">f"Slow traces (>5s): {</span><span class="tk-fn">len</span>(slow_traces)<span class="tk-str">}"</span>)</span>
    </div>
  </div>
  <div class="f2-explain">
    <div class="f2-explain-card" id="f2-explain-card"></div>
  </div>
</div>

</div>

<script>
var F2_DATA = [
  { color:'#FE3722', title:'Filter by Error Status',
    text:'Uses <code>filter_string="trace.status = \'ERROR\'"</code> to return only failed traces. The <code>trace.*</code> prefix is required for trace-level fields like <code>trace.status</code>, <code>trace.timestamp_ms</code>, and <code>trace.execution_time_ms</code>. You can also filter on <code>metadata.*</code>, <code>tag.*</code>, and <code>span.attributes.*</code> using <code>=</code>, <code>!=</code>, <code>&lt;</code>, <code>&gt;</code>, <code>LIKE</code>, and <code>AND</code>.' },
  { color:'#F8A805', title:'Filter by Execution Time',
    text:'Uses <code>filter_string="trace.execution_time_ms > 5000"</code> to find traces taking over 5 seconds. You can combine filters with <code>AND</code>, e.g. <code>trace.status = \'OK\' AND trace.execution_time_ms > 2500</code>.' }
];
var f2Current = null, f2Light = false;
function f2Toggle() {
  f2Light = !f2Light;
}
function f2Select(id) {
  var code = document.getElementById('f2-code'), card = document.getElementById('f2-explain-card');
  document.querySelectorAll('.f2-card').forEach(function(b){ b.classList.toggle('active', parseInt(b.dataset.id)===id); });
  if (f2Current === id) { code.classList.remove('has-highlight'); card.classList.remove('visible'); document.querySelectorAll('.f2-card').forEach(function(b){b.classList.remove('active');}); f2Current=null; return; }
  f2Current = id; var c = F2_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln){ ln.classList.toggle('hl', ln.dataset.g===String(id)); });
  card.style.borderTopColor = c.color;
  card.innerHTML = '<div style="font-size:15pt;font-weight:700;margin-bottom:10px;color:'+c.color+';">'+c.title+'</div><div>'+c.text+'</div>';
  card.classList.add('visible');
}
</script>

In [0]:
# Find error traces
error_traces = mlflow.search_traces(
    locations=[experiment_id],
    filter_string="trace.status = 'ERROR'",
    max_results=100
)
print(f"Error traces found: {len(error_traces)}")

# Find slow traces (over 5 seconds)
slow_traces = mlflow.search_traces(
    locations=[experiment_id],
    filter_string="trace.execution_time_ms > 5000",
    max_results=100
)
print(f"Slow traces (>5s): {len(slow_traces)}")

Error traces found: 1
Slow traces (>5s): 2


### E3. SQL-Based Trace Queries

For dashboard-scale analysis, you can also query traces stored in Unity Catalog tables using Databricks SQL (through the MLflow trace views).

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;"><code>mlflow.search_traces()</code>  is best for interactive, programmatic analysis and recent data, and it can optionally use a Databricks SQL warehouse for performance on large trace datasets. For BI dashboards and scheduled reporting, build queries on top of the Unity Catalog trace tables/views and consider materializing or optimizing them for large historical windows.</p>
</div>
</div>
</div>

## F. Conclusion

1. **Manual tracing** (`@mlflow.trace`) with semantic `span_type` labels adds visibility into custom logic: tool calls, retrieval, business logic
2. **Span attributes** attach production context (user ID, session ID, app version) that is critical for debugging
3. **Multi-turn tracing** uses a shared `session_id` attribute to group conversation turns
4. **Trace search** (`mlflow.search_traces()`) enables programmatic filtering by status, timing, and attributes
5. **SQL-based queries** on Unity Catalog trace views support dashboard-scale analysis for production monitoring

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
